# Probar el modelo Gemma-7b-it afinado en Vertex AI (Agent Platform -> Models -> Training)

Igual que `infer_gemma.ipynb`, pero para los adaptadores LoRA entrenados con
un **Custom Training Job** de Vertex AI / Gemini Enterprise Agent Platform
(el menú *Agent Platform -> Models -> Training*), en vez del entrenamiento
hecho dentro de esta misma VM+JupyterHub.

La diferencia real es una sola: el training job de Vertex AI no escribió los
adaptadores en el disco de esta VM -los escribió en un **bucket de Cloud
Storage** (la ruta que le pasaste como `output_dir` / `AIP_MODEL_DIR` al
job, algo como `gs://tu-bucket/gemma-7b-it-samsum-lora/`). Este notebook
descarga esa carpeta a un caché local con el cliente de Python
`google-cloud-storage` (no con `gcloud`/`gsutil`, que no viene instalado en
esta imagen) y de ahí en adelante usa exactamente la misma lógica de
generación que `infer_gemma.ipynb`.

Las credenciales para leer el bucket se toman automáticamente del servidor
de metadata de la VM (Application Default Credentials) -no hace falta
montar ningún archivo de service account, siempre que la cuenta de servicio
de la VM tenga permiso de lectura sobre el bucket (rol *Storage Object
Viewer* o superior).


## 0. Instalar el cliente de Cloud Storage

In [ ]:
%pip install --quiet google-cloud-storage


## 1. Configuración

Cambia `GCS_ADAPTER_DIR` por la ruta real del bucket donde quedó el
resultado del Custom Training Job (la verás en la consola de Agent Platform
-> Models -> Training, en los detalles del job, como "output directory").


In [ ]:
import os

GCS_ADAPTER_DIR = "gs://[tu-bucket]/gemma-7b-it-samsum-lora"   # <-- reemplaza esto
LOCAL_CACHE_DIR = "/home/jovyan/labs/gemma-7b-it-samsum-lora-vertex"
FORCE_DOWNLOAD = False   # cambia a True si volviste a entrenar y quieres refrescar el caché local

assert GCS_ADAPTER_DIR.startswith("gs://"), "GCS_ADAPTER_DIR debe empezar con 'gs://'"


## 2. Descargar los adaptadores desde Cloud Storage (con caché local)

In [ ]:
def download_gcs_dir(gcs_uri, local_dir, force=False):
    if os.path.isdir(local_dir) and os.listdir(local_dir) and not force:
        print(f"Ya existe una copia local en {local_dir}. Saltando descarga.")
        return local_dir

    from google.cloud import storage

    bucket_name, _, prefix = gcs_uri[len("gs://"):].partition("/")
    prefix = prefix.rstrip("/")

    print(f"Descargando {gcs_uri} -> {local_dir} ...")
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blobs = [b for b in bucket.list_blobs(prefix=prefix + "/" if prefix else prefix)
             if not b.name.endswith("/")]

    if not blobs:
        raise SystemExit(
            f"No encontré archivos en {gcs_uri}. Revisa que el Custom Training Job haya "
            "terminado exitosamente y que la ruta sea la correcta."
        )

    os.makedirs(local_dir, exist_ok=True)
    for blob in blobs:
        rel_path = blob.name[len(prefix):].lstrip("/") if prefix else blob.name
        dest = os.path.join(local_dir, rel_path)
        os.makedirs(os.path.dirname(dest) or local_dir, exist_ok=True)
        blob.download_to_filename(dest)
        print(f"  descargado: {rel_path}")

    print(f"Descarga completa: {len(blobs)} archivo(s) en {local_dir}")
    return local_dir


ADAPTER_DIR = download_gcs_dir(GCS_ADAPTER_DIR, LOCAL_CACHE_DIR, force=FORCE_DOWNLOAD)
print("Adaptadores disponibles en:", ADAPTER_DIR)
print(os.listdir(ADAPTER_DIR))


## 3. Login en Hugging Face (solo si hace falta)

Necesario solo si los pesos base de Gemma no quedaron ya en caché de una
sesión anterior de este contenedor.


In [ ]:
import getpass
from huggingface_hub import login

necesita_login = False  # cambia a True si te da un error 401/403 al cargar el modelo más abajo

if necesita_login:
    hf_token = getpass.getpass("Token de Hugging Face (con acceso a google/gemma-7b-it): ")
    login(token=hf_token)
    del hf_token


## 4. Cargar el modelo base (4-bit) + los adaptadores LoRA

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "google/gemma-7b-it"

print("CUDA disponible:", torch.cuda.is_available())

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True,
)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Memoria GPU ocupada tras cargar el modelo base (GB):",
      round(torch.cuda.memory_allocated() / 1e9, 2) if torch.cuda.is_available() else "N/A")

model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()
print("Adaptadores LoRA (entrenados en Vertex AI) cargados.")


## 5. Funciones auxiliares de generación

In [ ]:
def build_prompt(dialogue):
    messages = [
        {"role": "user", "content": f"Resume el siguiente diálogo en 1-2 frases:\n\n{dialogue}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def resumir(dialogue, max_new_tokens=64):
    prompt = build_prompt(dialogue)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def resumir_base(dialogue, max_new_tokens=64):
    """Igual que resumir(), pero desactivando el LoRA -para comparar contra
    el modelo base sin fine-tuning, sin tener que cargar dos copias del modelo."""
    with model.disable_adapter():
        return resumir(dialogue, max_new_tokens)


## 6. Ejemplos de prueba

Dos diálogos "de juguete" (no vistos en el entrenamiento) y uno al estilo
samsum (mensajes cortos e informales en inglés, como los que sí vio el
modelo durante el fine-tuning).


In [ ]:
ejemplos = [
    "Carlos: ¿Vas a venir a la reunión de las 3pm?\n"
    "Marta: Sí, ya salgo. ¿La sala sigue siendo la 402?\n"
    "Carlos: Sí, misma sala. Nos vemos ahí.",

    "Sofía: Se me quedó el cargador en tu casa ayer, ¿lo tienes ahí?\n"
    "Diego: Sí, lo vi en la mesa de la sala. Te lo llevo mañana a la oficina.\n"
    "Sofía: Perfecto, gracias!",

    "Ana: hey are we still on for the gym at 6?\n"
    "Leo: yeah but running a bit late, more like 6:20\n"
    "Ana: no worries, I'll grab a locker and wait",
]

for i, dialogo in enumerate(ejemplos, start=1):
    print(f"\n{'=' * 70}\nEjemplo {i}\n{'=' * 70}")
    print("Diálogo:\n" + dialogo)
    print("\n>> Resumen (modelo afinado con LoRA, entrenado en Vertex AI):\n" + resumir(dialogo))


## 7. Comparar contra el modelo base (sin fine-tuning)

In [ ]:
dialogo_prueba = ejemplos[0]

print("Diálogo:\n" + dialogo_prueba)
print("\n>> Con LoRA (afinado en Vertex AI):\n" + resumir(dialogo_prueba))
print("\n>> Sin LoRA (modelo base):\n" + resumir_base(dialogo_prueba))


## 8. Prueba con tu propio diálogo

In [ ]:
mi_dialogo = """Pega aquí tu propio diálogo, con un salto de línea por turno.
Persona A: ...
Persona B: ..."""

print(resumir(mi_dialogo))


## Notas finales

- Si te da un error de permisos al descargar del bucket (403), revisa que
  la cuenta de servicio de esta VM tenga el rol *Storage Object Viewer* (o
  superior) sobre el bucket -no sobre el proyecto entero, basta con el bucket.
- Si `GCS_ADAPTER_DIR` no es exactamente la ruta de salida del training job,
  la descarga falla con un mensaje claro ("No encontré archivos en ..."):
  copia la ruta desde los detalles del job en la consola de Agent Platform.
- Esta misma lógica existe como script plano en `infer_gemma_vertex.py`, con
  un flag `--compare_base` y `--gcs_adapter_dir` para correrlo de una sola vez.
